# 1. Project Introduction

# Mini Lakehouse Pipeline using PySpark & Delta Lake

## Project Overview

In this practical session, we will build a mini modern data engineering pipeline using PySpark and Delta Lake.

This project demonstrates how modern data platforms process, optimize, and manage large-scale analytical data.

---

## What We Will Learn

In this project, we will learn:

- How to ingest raw CSV data using PySpark
- How distributed DataFrames work
- How to optimize storage using Parquet format
- How Delta Lake provides ACID transactions
- How Time Travel allows querying older versions of data
- Introduction to real-time streaming using Spark Structured Streaming

---

## Technologies Used

| Component | Technology |
|---|---|
| Processing Engine | PySpark |
| Storage Optimization | Parquet |
| Lakehouse Layer | Delta Lake |
| Streaming | Spark Structured Streaming |
| Environment | Google Colab |
| Language | Python |

---

## Real-World Relevance

These concepts are widely used in modern data engineering systems such as:

- Databricks
- AWS EMR
- Azure Synapse
- Snowflake pipelines
- Real-time analytics systems

---

## Pipeline Architecture

Raw CSV Data
→ PySpark Ingestion
→ Data Cleaning
→ Parquet Optimization
→ Delta Lake Storage
→ Time Travel Queries
→ Streaming Demo

# 2. Environment Setup

# Step 1 — Environment Setup

Before building the pipeline, we need to install:

- PySpark → Distributed data processing engine
- Delta Lake → Lakehouse storage layer with ACID support

These libraries allow us to simulate modern data engineering workflows directly inside Google Colab.

In [ ]:
# Install PySpark
!pip install pyspark -q

# Install Delta Lake
!pip install delta-spark -q

 Creating Spark Session

A SparkSession is the main entry point for working with Apache Spark.

It allows us to:
- Read datasets
- Process distributed data
- Execute transformations
- Work with Delta Lake

In [ ]:
# Step 1: Kill the existing session
from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession()
if spark:
    spark.stop()
    print("Old session stopped")

In [ ]:
# Step 2: Create a fresh session with Delta
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("MiniLakehousePipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(spark.version)
print("Delta-enabled Spark Session created")

4.0.2
Delta-enabled Spark Session created



# 3. Dataset Loading


## Step 3.1 — Uploading the Dataset


In this section, we will load the raw CSV dataset into PySpark.

The dataset contains e-commerce sales transaction records that will be used throughout the pipeline.

---

# Step 3.1 — Uploading the Dataset

First, we upload the CSV file into Google Colab.

Google Colab provides temporary storage during the notebook session, allowing us to work with local files directly.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving ecommerce_sales_34500.csv to ecommerce_sales_34500 (1).csv


# Step 3.2 — Reading the CSV File using PySpark

PySpark can read structured datasets such as CSV files into distributed DataFrames.

We use:
- `header=True` → tells Spark that the first row contains column names
- `inferSchema=True` → automatically detects data types

This allows Spark to understand the structure of the dataset automatically.

In [ ]:
df = spark.read.csv(
    "ecommerce_sales_34500.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


# Step 3.3 — Viewing Sample Records

After loading the dataset, we inspect a few rows to verify that the data has been loaded correctly.

The `show()` function displays sample records from the DataFrame.

In [ ]:
df.show(5)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
| O100000|     C17270|   P234890|       Home|164.08|    0.15|       1|   Credit Card|2023-12-23|                 4|  West|      No|      139.47|         7.88|        31.17|          60|         Female|
| O100001|     C17603|   P228204|    Grocery| 24.73|     0.0|       1|   Credit Card|2025-04-03|                 6| South|      No|       24.73|          4.6|        -2.62|          37|       

# Step 3.4 — Understanding the Dataset Schema

Schema represents the structure of the dataset.

It includes:
- Column names
- Data types
- Nullable information

Understanding schema is important because Spark uses it for distributed processing and query optimization.

In [ ]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_time_days: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- profit_margin: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_gender: string (nullable = true)



# Step 3.5 — Counting Total Records

Before processing the data, it is useful to know the dataset size.

The `count()` function returns the total number of rows in the DataFrame.

In [ ]:
row_count = df.count()

print(f"Total Rows in Dataset: {row_count}")

Total Rows in Dataset: 34500


# Step 3.6 — Viewing Column Names

We can inspect all available columns in the dataset using the `columns` attribute.

This helps us understand what information is available for analysis and transformation.

In [ ]:
print(df.columns)

['order_id', 'customer_id', 'product_id', 'category', 'price', 'discount', 'quantity', 'payment_method', 'order_date', 'delivery_time_days', 'region', 'returned', 'total_amount', 'shipping_cost', 'profit_margin', 'customer_age', 'customer_gender']


# 4. Data Inspection

Before transforming data, data engineers first inspect and analyze the dataset.

Data inspection helps us:
- Identify missing values
- Detect incorrect records
- Understand data distribution
- Validate data quality
- Prepare data for analytics

This is an important step in real-world data engineering pipelines.

# Step 4.2 — Selecting Specific Columns

Sometimes we only need a few columns instead of the entire dataset.

The `select()` function allows us to retrieve specific columns from the DataFrame.

In [ ]:
df.select("product_id", "category","total_amount","discount").show(5)

+----------+-----------+------------+--------+
|product_id|   category|total_amount|discount|
+----------+-----------+------------+--------+
|   P234890|       Home|      139.47|    0.15|
|   P228204|    Grocery|       24.73|     0.0|
|   P213892|Electronics|       166.8|    0.05|
|   P208689|Electronics|       63.67|     0.0|
|   P228063|       Home|       13.88|    0.15|
+----------+-----------+------------+--------+
only showing top 5 rows


In [ ]:
print(df.columns)

['order_id', 'customer_id', 'product_id', 'category', 'price', 'discount', 'quantity', 'payment_method', 'order_date', 'delivery_time_days', 'region', 'returned', 'total_amount', 'shipping_cost', 'profit_margin', 'customer_age', 'customer_gender']


# Step 4.3 — Filtering Records

Filtering allows us to retrieve rows that match specific conditions.

This is commonly used for:
- business analysis
- data cleaning
- anomaly detection
- customer segmentation

In [ ]:
df.filter(df["total_amount"] > 500).show(5)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days| region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+
| O100018|     C17831|   P218405|Electronics|734.32|     0.0|       2|        Wallet|2025-06-20|                 4|  South|      No|     1468.64|         10.9|       165.34|          58|         Female|
| O100024|     C15611|   P247930|     Sports|345.57|    0.15|       2|    Debit Card|2024-04-16|                 5|Central|      No|      587.47|         9.48|       166.76|          32|  

# Step 4.4 — Generating Descriptive Statistics

The `describe()` function provides summary statistics for numerical columns.

This includes:
- count
- mean
- standard deviation
- minimum value
- maximum value

These statistics help us understand the dataset distribution.

In [ ]:
df.describe().show()

+-------+--------+-----------+----------+--------+------------------+-------------------+------------------+--------------+------------------+-------+--------+------------------+------------------+-----------------+------------------+---------------+
|summary|order_id|customer_id|product_id|category|             price|           discount|          quantity|payment_method|delivery_time_days| region|returned|      total_amount|     shipping_cost|    profit_margin|      customer_age|customer_gender|
+-------+--------+-----------+----------+--------+------------------+-------------------+------------------+--------------+------------------+-------+--------+------------------+------------------+-----------------+------------------+---------------+
|  count|   34500|      34500|     34500|   34500|             34500|              34500|             34500|         34500|             34500|  34500|   34500|             34500|             34500|            34500|             34500|          345

# Step 4.5 — Checking Missing Values

Missing values are common in real-world datasets.

Null values can:
- affect analytics
- break transformations
- produce incorrect results

Data engineers must identify missing data before processing.

In [ ]:
from pyspark.sql.functions import col, when, count, lower

df.select([
    count(
        when(
            col(c).isNull() | (lower(col(c)) == "null"),
            c
        )
    ).alias(c)
    for c in df.columns
]).show()

+--------+-----------+----------+--------+-----+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|category|price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+--------+-----+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|       0|          0|         0|       0|    0|       0|       0|             0|         0|                 0|     0|       0|           0|            0|            0|           0|              0|
+--------+-----------+----------+--------+-----+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+



# Step 4.6 — Checking Duplicate Records

Duplicate records can create incorrect analytics and reporting issues.

We compare:
- total row count
- distinct row count

to identify duplicate records.

In [ ]:
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total Rows:", total_rows)
print("Distinct Rows:", distinct_rows)
print("Duplicate Rows:", total_rows - distinct_rows)

Total Rows: 34500
Distinct Rows: 34500
Duplicate Rows: 0


# Step 4.7 — Grouping and Aggregation

Aggregation helps summarize business information.

Examples:
- total sales by category
- total orders by region
- average revenue per product

This is widely used in analytics and reporting systems.

In [ ]:
from pyspark.sql.functions import sum

df.groupBy("Category") \
  .agg(sum("total_amount").alias("Total_Sales")) \
  .show()

+-----------+------------------+
|   Category|       Total_Sales|
+-----------+------------------+
|       Home|1077681.5200000005|
|    Fashion| 471545.8000000008|
|     Sports| 629825.5400000004|
|    Grocery| 82000.51000000007|
|Electronics| 3319206.500000002|
|     Beauty|153019.37999999957|
|       Toys|132013.79999999993|
+-----------+------------------+



# Step 4.8 — Sorting Data

Sorting helps organize records in ascending or descending order.

This is useful for:
- ranking
- reporting
- identifying top-performing products or customers

In [ ]:
df.orderBy(df["total_amount"].desc()).show(5)

+--------+-----------+----------+-----------+-------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|   category|  price|discount|quantity|payment_method|order_date|delivery_time_days| region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+-----------+-------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+
| O120765|     C16655|   P217031|Electronics|2586.36|     0.0|       5|    Debit Card|2024-02-22|                 5|   West|      No|     12931.8|        15.65|      1536.17|          28|           Male|
| O107705|     C13565|   P242326|Electronics|2930.47|     0.0|       4|           UPI|2024-04-13|                 4|   West|      No|    11721.88|        13.67|      1392.96|          

# Step 4.9 — Understanding Business Metrics

The dataset contains multiple business-related fields used in analytics systems.

Important columns include:

- `total_amount` → Total transaction revenue
- `profit_margin` → Profit earned from the order
- `discount` → Discount applied
- `shipping_cost` → Delivery cost
- `returned` → Whether the item was returned
- `delivery_time_days` → Delivery duration

These fields help organizations analyze:
- profitability
- operational efficiency
- customer behavior
- regional performance

# Step 4.10 — Revenue Analysis by Category

We can aggregate transaction revenue to identify which product categories generate the highest revenue.

This type of analysis is commonly used in:
- business intelligence dashboards
- executive reporting
- sales analytics systems

In [ ]:
from pyspark.sql.functions import sum

df.groupBy("category") \
  .agg(sum("total_amount").alias("total_revenue")) \
  .orderBy("total_revenue", ascending=False) \
  .show()

+-----------+------------------+
|   category|     total_revenue|
+-----------+------------------+
|Electronics| 3319206.500000002|
|       Home|1077681.5200000005|
|     Sports| 629825.5400000004|
|    Fashion| 471545.8000000008|
|     Beauty|153019.37999999957|
|       Toys|132013.79999999993|
|    Grocery| 82000.51000000007|
+-----------+------------------+



# 5. Data Cleaning

Real-world datasets often contain:
- missing values
- duplicate records
- incorrect data types
- inconsistent values

Data cleaning is an important part of every data engineering pipeline.

Even though our dataset is already clean, we will still perform common cleaning operations to understand the standard workflow used in production systems.

## Step 5.2 — Removing Duplicate Records

Duplicate records can lead to:
- incorrect analytics
- inflated revenue calculations
- inaccurate reporting

The `dropDuplicates()` function removes repeated rows from the dataset.

In [ ]:
clean_df = df.dropDuplicates()

print("Duplicates Removed Successfully")

Duplicates Removed Successfully


# Step 5.3 — Verifying Record Count

After removing duplicates, we verify the total number of records again.

In [ ]:
print("Original Row Count:", df.count())
print("Cleaned Row Count:", clean_df.count())

Original Row Count: 34500
Cleaned Row Count: 34500


# Step 5.4 — Handling Missing Values

Missing values are common in production datasets.

Spark provides multiple ways to handle null values:
- remove rows
- replace values
- fill default values

In this dataset, no missing values exist, but we will still demonstrate the workflow.

In [ ]:
clean_df = clean_df.fillna({
    "discount": 0,
    "shipping_cost": 0
})

print("Missing Value Handling Completed")

Missing Value Handling Completed


# Step 5.5 — Validating Data Types

Correct data types are important for:
- calculations
- filtering
- aggregations
- query optimization

We inspect the schema again after cleaning.

In [ ]:
clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_time_days: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = false)
 |-- profit_margin: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_gender: string (nullable = true)



# Step 5.6 — Creating Derived Columns

Data engineers often create derived columns for analytics purposes.

Here we calculate:

`final_price_after_shipping`

This represents:
- transaction amount
- plus shipping cost

In [ ]:
from pyspark.sql.functions import col

clean_df = clean_df.withColumn(
    "final_price_after_shipping",
    col("total_amount") + col("shipping_cost")
)

clean_df.select(
    "total_amount",
    "shipping_cost",
    "final_price_after_shipping"
).show(5)

+------------+-------------+--------------------------+
|total_amount|shipping_cost|final_price_after_shipping|
+------------+-------------+--------------------------+
|       50.52|          6.2|        56.720000000000006|
|       16.35|         4.57|                     20.92|
|       12.82|         3.74|        16.560000000000002|
|      413.13|         8.79|                    421.92|
|        73.6|         6.89|                     80.49|
+------------+-------------+--------------------------+
only showing top 5 rows


# Step 5.7 — Filtering Invalid Records

In real-world systems, some records may contain invalid values.

Examples:
- negative prices
- zero quantities
- impossible delivery times

We filter datasets to retain only valid business records.

In [ ]:
clean_df = clean_df.filter(col("quantity") > 0)

print("Invalid Record Filtering Completed")

Invalid Record Filtering Completed


# Step 5.8 — Final Clean Dataset

The dataset has now passed through:
- duplicate handling
- null handling
- validation
- derived transformations

This cleaned dataset will be used for:
- optimization
- Delta Lake storage
- analytics processing

In [ ]:
clean_df.show(5)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+--------------------------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days| region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|final_price_after_shipping|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+--------------------------+
| O100019|     C10130|   P248629|    Fashion| 50.52|     0.0|       1|   Credit Card|2024-10-10|                 3|  North|     Yes|       50.52|          6.2|        11.48|          56|           Male|        56.720000000000006|
| O100131|     C17208|   P230083|       Toys|  5.45|     0.0|       3|        Wa

# 6. Parquet Optimization

CSV files are simple and easy to read, but they are not optimized for analytics workloads.

Modern data engineering systems use columnar storage formats such as Parquet.

---

## Why Parquet is Important

Parquet provides:
- columnar storage
- compression
- faster query performance
- reduced storage usage

This makes Parquet highly suitable for:
- data lakes
- analytics platforms
- reporting systems
- big data processing

---

## CSV vs Parquet

| CSV | Parquet |
|---|---|
| Row-based | Column-based |
| Larger file size | Compressed storage |
| Slower analytics | Faster analytics |
| No schema optimization | Optimized schema handling |

# Step 6.2 — Converting CSV Data into Parquet

We now convert the cleaned dataset into Parquet format.

Spark will:
- optimize storage
- preserve schema
- store the dataset efficiently

In [ ]:
clean_df.write.mode("overwrite").parquet("sales_parquet")

# Step 6.3 — Reading the Parquet Dataset

We now load the Parquet dataset back into Spark.

This demonstrates how optimized analytical datasets are read in modern data platforms.

In [ ]:
parquet_df = spark.read.parquet("sales_parquet")

parquet_df.show(5)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+--------------------------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|final_price_after_shipping|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+--------------------------+
| O100268|     C13763|   P207405|Electronics|309.52|     0.0|       1|           UPI|2024-08-15|                 5|  East|      No|      309.52|         8.23|        28.91|          20|         Female|                    317.75|
| O100298|     C11707|   P217221|Electronics|157.79|     0.0|       1|           UPI

# Step 6.4 — Comparing CSV and Parquet Performance

We now compare the read performance between:
- raw CSV files
- optimized Parquet files

Parquet is usually faster because:
- only required columns are scanned
- data is compressed
- metadata is optimized

In [ ]:
import time

In [ ]:
# CSV Read Timing
start_time = time.time()

csv_df = spark.read.csv(
    "ecommerce_sales_34500.csv",
    header=True,
    inferSchema=True
)

csv_df.count()

csv_time = time.time() - start_time

print(f"CSV Read Time: {csv_time:.2f} seconds")

CSV Read Time: 0.80 seconds


In [ ]:
# Parquet Read Timing

start_time = time.time()

parquet_df = spark.read.parquet("sales_parquet")

parquet_df.count()

parquet_time = time.time() - start_time

print(f"Parquet Read Time: {parquet_time:.2f} seconds")

Parquet Read Time: 0.66 seconds


# Step 6.5 — Understanding Columnar Storage

Parquet stores data column-by-column instead of row-by-row.

This improves analytics performance because Spark can:
- scan only required columns
- reduce disk reads
- improve aggregation performance

This optimization is very important in big data systems.

# Step 6.6 — Running Analytics on Parquet Data

We can now perform analytical queries directly on the optimized Parquet dataset.

In [ ]:
from pyspark.sql.functions import avg

parquet_df.groupBy("category") \
    .agg(avg("profit_margin").alias("average_profit")) \
    .orderBy("average_profit", ascending=False) \
    .show()

+-----------+-------------------+
|   category|     average_profit|
+-----------+-------------------+
|Electronics|  55.72358737864076|
|       Home|  47.86471660287957|
|     Sports|  38.48511388156318|
|    Fashion| 20.597161816437513|
|     Beauty| 11.990394833048995|
|       Toys| 7.9277725453261105|
|    Grocery|-2.2641596845736798|
+-----------+-------------------+



# 7. Delta Lake Implementation

Delta Lake is a modern storage layer built on top of data lakes.

It combines:
- scalability of data lakes
- reliability of databases

Delta Lake provides important enterprise features such as:
- ACID transactions
- schema enforcement
- version history
- Time Travel
- reliable batch and streaming support

---

## Why Delta Lake Matters

Traditional data lakes using raw CSV or Parquet files can face problems such as:
- inconsistent data
- corrupted writes
- lack of versioning
- difficult updates and deletes

Delta Lake solves these problems using transaction logs and metadata management.

# Step 7.2 — Creating a Delta Lake Table

We now store the cleaned dataset using Delta Lake format.

Spark will create:
- optimized Parquet files
- Delta transaction logs
- metadata tracking files

This creates a transactional lakehouse table.

In [ ]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("delta/sales")

# Step 7.3 — Reading the Delta Table

We now load the Delta table back into Spark.

Even though Delta uses Parquet internally, Spark also reads:
- transaction logs
- metadata
- version history

In [ ]:
delta_df = spark.read.format("delta").load("delta/sales")

delta_df.show(5)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+--------------------------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|final_price_after_shipping|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+--------------------------+
| O100268|     C13763|   P207405|Electronics|309.52|     0.0|       1|           UPI|2024-08-15|                 5|  East|      No|      309.52|         8.23|        28.91|          20|         Female|                    317.75|
| O100298|     C11707|   P217221|Electronics|157.79|     0.0|       1|           UPI

# Step 7.4 — Understanding Delta Transaction Logs

Delta Lake maintains a special folder called `_delta_log`.

This folder stores:
- transaction history
- metadata
- schema information
- table versions

This is what enables:
- ACID transactions
- rollback capability
- Time Travel queries

In [ ]:
import os

os.listdir("delta/sales")

['_delta_log',
 'part-00000-551b19c7-0b4d-4e6c-827b-689443932d8e-c000.snappy.parquet',
 '.part-00000-551b19c7-0b4d-4e6c-827b-689443932d8e-c000.snappy.parquet.crc',
 '.part-00001-ca7c9309-528d-4075-9bd9-eb906927bbdc-c000.snappy.parquet.crc',
 'part-00001-ca7c9309-528d-4075-9bd9-eb906927bbdc-c000.snappy.parquet']

# Step 7.5 — Updating the Dataset

In real systems, data changes over time.

Examples:
- order updates
- refunds
- customer corrections
- inventory changes

We now simulate an update operation to create a new Delta table version.

In [ ]:
updated_df = delta_df.withColumn(
    "shipping_cost",
    col("shipping_cost") + 2
)

# NEW SHIPPING COST = OLD SHIPPING COST + 2

# Step 7.6 — Creating a New Delta Version

We now overwrite the Delta table with updated data.

Delta Lake automatically creates:
- a new table version
- a new transaction entry
- updated metadata

In [ ]:
updated_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("delta/sales")


# Step 7.7 — Reading the Latest Delta Version

The latest version now contains updated shipping costs.

In [ ]:
latest_df = spark.read.format("delta").load("delta/sales")

latest_df.select(
    "shipping_cost"
).show(5)



+-------------+
|shipping_cost|
+-------------+
|          6.2|
|         4.57|
|         3.74|
|         8.79|
|         6.89|
+-------------+
only showing top 5 rows


# 8. Time Travel in Delta Lake

Time Travel allows us to query previous versions of a dataset.

This is possible because Delta Lake maintains:
- transaction logs
- version history of every write operation

---

## Why This Matters

In real systems, Time Travel is used for:
- data recovery
- auditing changes
- debugging pipelines
- reproducibility of analytics

This is a core advantage of Lakehouse architecture.

In [ ]:
# Viewing Delta Table History

from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "delta/sales")

delta_table.history().show()

+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2026-05-19 12:17:...|  NULL|    NULL|    WRITE|{mode -> Overwrit...|NULL|    NULL|     NULL|          0|  Serializable|        false|{numFiles -> 2, n...|        NULL|Apache-Spark/4.0....|
|      0|2026-05-19 12:17:...|  NULL|    NULL|    WRITE|{mode -> Overwrit...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 2, n...|        NULL|Apache-Spark/4.0....|
+-------+-

# Step 8.3 — Time Travel Query

We can query an older version of the dataset using:

- version number
- or timestamp

This allows us to "go back in time" and see previous states of the data.

In [ ]:
old_version_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("delta/sales")

old_version_df.select("shipping_cost").show(5)

+-------------+
|shipping_cost|
+-------------+
|         8.23|
|         6.55|
|         3.36|
|         6.94|
|         6.39|
+-------------+
only showing top 5 rows


# Step 8.4 — Comparing Versions

We now compare:
- original dataset (version 0)
- updated dataset (latest version)

This demonstrates how data evolves over time in a lakehouse system.

In [ ]:
print("Old Version")
old_version_df.select("order_id", "shipping_cost").filter("order_id IN ('O100268', 'O100298', 'O100432', 'O100486', 'O101343')").orderBy("order_id").show()

print("Latest Version")
latest_df.select("order_id", "shipping_cost").filter("order_id IN ('O100268', 'O100298', 'O100432', 'O100486', 'O101343')").orderBy("order_id").show()

Old Version
+--------+-------------+
|order_id|shipping_cost|
+--------+-------------+
| O100268|         8.23|
| O100298|         6.55|
| O100432|         3.36|
| O100486|         6.94|
| O101343|         6.39|
+--------+-------------+

Latest Version
+--------+-----------------+
|order_id|    shipping_cost|
+--------+-----------------+
| O100268|            10.23|
| O100298|             8.55|
| O100432|5.359999999999999|
| O100486|8.940000000000001|
| O101343|             8.39|
+--------+-----------------+



# Step 8.5 — ACID Properties in Delta Lake

Delta Lake ensures:

- Atomicity → complete write or nothing
- Consistency → valid data state
- Isolation → concurrent safety
- Durability → data is never lost

This is what makes Delta Lake different from raw Parquet storage.

# 9. Streaming with Spark Structured Streaming

So far, we worked with batch data (static datasets).

Now we introduce the concept of streaming data.

---

## What is Streaming?

Streaming means:
- data arrives continuously
- system processes data in real-time or near real-time

---

## Real-world examples:
- order events in e-commerce
- payment transactions
- IoT sensor data
- ride-hailing updates (Uber/Lyft)

---

## In this demo:
We simulate streaming using Spark's built-in rate source.

In [ ]:
stream_df = spark.readStream.format("rate").load()

stream_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



# Step 9.3 — Viewing Streaming Data

The `rate` source generates:
- timestamp
- incremental value

This simulates a real-time event stream.

In [ ]:
query = stream_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

# Step 9.4 — Running the Stream

The stream will continuously output data.

To stop it:
- click interrupt kernel
- or use `query.stop()`

In [ ]:
import time

time.sleep(10)

query.stop()

# Step 9.5 — Streaming in Real Systems

In real production systems:

Streaming data would come from:
- Kafka topics
- event buses
- application logs
- IoT devices

Spark Structured Streaming processes this data incrementally and writes it into:
- Delta Lake tables
- data warehouses
- analytics systems

# 10. Conclusion

In this project, we built a mini data lakehouse pipeline using PySpark and Delta Lake.

The pipeline demonstrated a complete data engineering workflow from raw data ingestion to advanced lakehouse features.

---

## What We Implemented

- Data ingestion using PySpark
- Data inspection and validation
- Data cleaning and transformation
- Storage optimization using Parquet
- Lakehouse implementation using Delta Lake
- ACID transactions and metadata tracking
- Time Travel (versioning system)
- Basic streaming simulation

# Architecture Overview

The system follows a modern lakehouse architecture:

```
                ┌─────────────────────┐
                │   Raw CSV Dataset   │
                └─────────┬───────────┘
                          ↓
                ┌─────────────────────┐
                │   PySpark Ingestion │
                └─────────┬───────────┘
                          ↓
                ┌─────────────────────┐
                │   Data Cleaning     │
                └─────────┬───────────┘
                          ↓
                ┌─────────────────────┐
                │ Parquet Optimization│
                └─────────┬───────────┘
                          ↓
                ┌─────────────────────┐
                │   Delta Lake Table  │
                └─────────┬───────────┘
                          ↓
        ┌────────────────────────────────────┐
        │ Time Travel / Versioning (ACID)    │
        └────────────────────────────────────┘
                          ↓
        ┌────────────────────────────────────┐
        │ Structured Streaming (Simulation) │
        └────────────────────────────────────┘
```

# Key Learning Outcomes

After completing this project, students should understand:

## Data Engineering Fundamentals
- ETL pipeline design
- schema handling
- data transformation logic

## Storage Optimization
- Parquet vs CSV
- columnar storage benefits

## Lakehouse Architecture
- Delta Lake concept
- ACID compliance in data lakes
- version control in datasets

## Advanced Concepts
- Time Travel
- incremental updates
- streaming data processing basics

# Real-World Relevance

This architecture is similar to systems used in:

- Databricks Lakehouse
- AWS data pipelines (S3 + Glue + Athena)
- Azure Synapse Analytics
- BigQuery data pipelines

---

## Industry Use Cases

- E-commerce analytics
- financial transaction processing
- logistics tracking systems
- real-time monitoring systems

# Final Note

This project demonstrates how raw data evolves into a structured, optimized, and version-controlled lakehouse system.

It serves as a foundation for understanding modern data engineering systems used in production environments.